## Fst and spatial population genetic structure

We want to know the extent to which our populations are genetically separate from one another. While definitively proving that populations are reproductively isolated from one another is impossible with the data available, we can identify barriers and corridors to gene flow in the landscape, and the extent to which our populations represent discrete or continuous groups.

We can start by using Fst to characterise geneti differentiation across the landscape.

In [8]:
# Load libraries
import cairn as cn
import allel
import pandas as pd
import numpy as np
import zarr

from yaspin import yaspin

from cairn.utils import parse_region, locate_region, hash_params, hash_columns
from cairn.fst import fst_average, fst_gwss # load modules for calculating average and windowed fst

from Bio import SeqIO

In [5]:
# Load external data that we need

# Path to the genotype data
zarr_p = "/scratch/user/uqtdenni/afar_production_bunya/curation/uq-beebe-001/staged_zarr/{contig}.zarr/"

# Get contigs
ref_path = '/scratch/user/uqtdenni/afar_production_bunya/reference/VectorBase-54_AfarautiFAR1_Genome.fasta'
# now let's get a list of the contigs that we are going to call over
contig_lengths = {}
for record in SeqIO.parse(ref_path, "fasta"):
    seq_id = record.id
    seq_length = len(record.seq)
    contig_lengths[seq_id] = seq_length
filtered_contigs = {k: v for k, v in sorted(contig_lengths.items(), key=lambda item: item[1], reverse=True) if v > 100000}

# Now load some interesting genes
#df of gene locations
gene_locs = pd.read_csv('/scratch/user/uqtdenni/far_hin_1.x/work/pop_gen_2025/gene_locs.txt', sep = '\t')
gene_locs

,GeneName,Afar_id,contig,start,end,yval
0,Cyp6aa1,AFAF002114,KI915040,28103725,28106779,1.1
1,Coeae,AFAF003084,KI915046,387367,389267,1.1
2,Gste,AFAF021739,KI915043,9035514,9036691,1.1
3,Ace1,AFAF012446,KI915040,3467460,3482203,1.1
4,Vgsc,AFAF014542,KI915041,10621202,10641255,1.1
5,Tep1,AFAF019347,KI915052,1398259,1404046,1.1
6,P47R,AFAF006142,KI915044,1163455,1165201,1.1
7,Cyp9k1,AFAF009018,KI915045,1866712,1868829,1.1
8,Orco,AFAF009348,KI915040,238151,247096,1.1


In [6]:
# Import metadata
df_samples = pd.read_csv('/scratch/user/uqtdenni/far_hin_1.x/work/metadata_development_20250702/metadata-staged-cohorts-20251119.txt',sep='\t')
df_samples.head()

,Unnamed: 0.1,sample_id,index,Unnamed: 0,origin,taxon,site,UIN (if applicable),latitude,longitude,...,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,analysis_cohort
0,0,far8_98_89_10,0,0,NaN,farauti_8,NaN,NaN,-7.748,146.43,...,49.609203,6.675676,-3.857840,200.24861,-458.60718,3.157878,-8.609264,5.877584,1.262067,farauti_8
1,1,far8_98_89_18,1,1,NaN,farauti_8,NaN,NaN,-7.748,146.43,...,49.145832,5.473371,-3.861486,191.36034,-430.16986,2.113821,-7.848210,5.508193,2.020197,farauti_8
2,2,far8_98_89_25,2,2,NaN,farauti_8,NaN,NaN,-7.748,146.43,...,50.703720,6.557748,-4.140456,206.63995,-486.98514,3.887656,-8.666692,7.122447,1.266567,farauti_8
3,3,far8_98_89_26,3,3,NaN,farauti_8,NaN,NaN,-7.748,146.43,...,48.763012,6.569715,-3.762237,196.97585,-448.15480,3.223184,-7.374317,6.238574,1.398220,farauti_8
4,4,far8_98_89_27,4,4,NaN,farauti_8,NaN,NaN,-7.748,146.43,...,50.387910,6.556661,-3.731082,203.00461,-476.17860,3.758885,-8.397419,7.258868,1.767493,farauti_8


We are going to start by iterating over pairs of population groupings (see the slideshow I have sent to you previously) for farauti ss and calculating genome-wide Fst between them.

In [12]:
# Now, we are going to iterate over our populations and calculate genome-wide Fst between them.
# As we are only interested in farauti sensu stricto, we want to subset our data to include only this taxon.
# Select unique analysis cohort identifiers for farauti-ss
farauti_cohorts = df_samples.query('taxon_pca == "farauti_ss"').analysis_cohort.unique()
farauti_cohorts

array(['farauti_ss-Bou', 'farauti_ss-Van', 'farauti_ss-Gua',
       'farauti_ss-MRL', 'farauti_ss-AUNT', 'farauti_ss-QLD-Mainland',
       'farauti_ss-TSI-sNG', 'farauti_ss-MK', 'farauti_ss-WP',
       'farauti_ss-NPP', 'farauti_ss-nNG', 'farauti_ss-MBA',
       'farauti_ss-SPP'], dtype=object)

In [4]:
# Here is an example of how to call the fst_average function
# This calculates the average Fst across a genomic region, as well as the standard error computed using block-jackknife
# We are doing this for a pair of populations

popa = 'farauti_ss-QLD-Mainland'
popb = 'farauti_ss-TSI-sNG'

fst, se = fst_average(
    sample_query_a = f'analysis_cohort == "{popa}"',
    sample_query_b = f'analysis_cohort == "{popb}"',
    region="KI915045",
    df_samples = df_samples,
    block_length = 2000,
    zarr_base_path = zarr_p,
)

In [7]:
fst

np.float64(0.2121506375880835)

In [5]:
# Load genotypes

# Query A
# Query B

# Create allele counts for both

# Load position data

# infer fst in windows

# clip to be zero at min


def fst_gwss(
    sample_query_a: str,
    sample_query_b: str,
    window_size: int,
    contig: str,
    zarr_base_path: str, 
    df_samples: pd.DataFrame, 
    clip_min : float | int = 0,
    genotype_var: str = "calldata/GT",
    pos_var: str = "variants/POS",
    filter_mask: str = 'variants/filter_pass',
    results_dir: str ='results_cache/results_fst_v1',
    overwrite: bool = False,
    ) -> pd.DataFrame: 

    """"
    Perform genome scan of windowed Hudson's Fst across a specific genomic region (contig or region).
    """

    # Set up params for hashing
    params = dict(
            sample_query_a = sample_query_a,
            sample_query_b = sample_query_b,
            contig=contig,
            window_size=window_size,
        )

    results_key = hash_params(
        params
    )



    # define paths for results files
    fst_path = f'{results_dir}/{results_key}-fst.npy'
    x_path = f'{results_dir}/{results_key}-x.npy'


    # Hashing or not
    if overwrite is False:
        try:
            # try to load previously generated results
            fst = np.load(fst_path)
            x = np.load(x_path)
            return (fst, x)
        except FileNotFoundError:
            # no previous results available, need to run analysis
            print(f'running analysis: {results_key}')

    # Load genos for query a
    ac_a = load_genotype_array(
            zarr_base_path=zarr_base_path,
            region=contig,
            df_samples=df_samples,
            genotype_var=genotype_var,
            pos_var=pos_var,
            sample_query=sample_query_a,
            filter_mask=filter_mask,
        ).count_alleles()

    # Load genos for query b
    ac_b = load_genotype_array(
            zarr_base_path=zarr_base_path,
            region=contig,
            df_samples=df_samples,
            genotype_var=genotype_var,
            pos_var=pos_var,
            sample_query=sample_query_b,
            filter_mask=filter_mask,
        ).count_alleles()

    
    # Get pos data: TODO -refactor this into utils
    z = zarr.open(zarr_base_path.format(contig=contig))

    # Get variant position array
    pos = z["variants/POS"]
    flt = z["variants/filter_pass"][:]
    pos = pos[flt]

    with yaspin("Compute Fst..."):
        with np.errstate(divide="ignore", invalid="ignore"):
            fst = allel.moving_hudson_fst(ac_a, ac_b, size=window_size)
            # Sometimes Fst can be very slightly below zero, clip for simplicity.
            fst = np.clip(fst, a_min=clip_min, a_max=1)
            x = allel.moving_statistic(pos, statistic=np.mean, size=window_size)

    # Save outputs
    os.makedirs(results_dir, exist_ok=True)
    np.save(fst_path, fst)
    np.save(x_path, x)

    print(f'saved results: {results_key}')

    return (fst, x)


class GenomePositionMapper:
    def __init__(self, contig_lengths_dict):
        """
        contig_lengths_dict: dict {contig_id: length}, already sorted longest first
        """
        self.sorted_contigs = list(contig_lengths_dict.keys())
        sorted_lengths = [contig_lengths_dict[c] for c in self.sorted_contigs]

        # compute offsets
        contig_offsets = np.cumsum([0] + sorted_lengths[:-1])
        self.contig_offsets = dict(zip(self.sorted_contigs, contig_offsets))

        # zebra striping
        self.color_map = {
            contig: ("lightblue" if i % 2 == 0 else "steelblue")
            for i, contig in enumerate(self.sorted_contigs)
        }

    def _apply(self, df, coord):
        df = df.copy()
        df["contig"] = pd.Categorical(df["contig"], categories=self.sorted_contigs, ordered=True)
        df = df.sort_values(["contig", coord])
        df["contig_offset"] = df["contig"].map(self.contig_offsets).astype(float)
        df["genome_position"] = df[coord] + df["contig_offset"]
        df["color"] = df["contig"].map(self.color_map)
        return df

    def transform_positions(self, df):
        return self._apply(df, "pos")

    def transform_annotations(self, df):
        return self._apply(df, "start")


In [6]:
# Import metadata
df_samples = pd.read_csv('/scratch/user/uqtdenni/far_hin_1.x/work/metadata_development_20250702/metadata-staged-cohorts-20251119.txt',sep='\t')
zarr_p = "/scratch/user/uqtdenni/afar_production_bunya/curation/uq-beebe-001/staged_zarr/{contig}.zarr/"

In [10]:
df_samples.analysis_cohort.unique()

array(['farauti_8', 'farauti_ss-Bou', 'farauti_ss-Van', 'farauti_ss-Gua',
       'farauti_ss-MRL', 'farauti_ss-AUNT', 'farauti_ss-QLD-Mainland',
       'farauti_ss-TSI-sNG', 'farauti_ss-MK', 'farauti_ss-WP',
       'hinesorum-WP', 'farauti_ss-NPP', 'farauti_ss-nNG',
       'farauti_ss-MBA', 'farauti_ss-SPP', 'hinesorum-Bou',
       'hinesorum-Gua', 'hinesorum-PP', 'hinesorum-QLD',
       'hinesorum-unassigned', 'q-oreios-highlands', 'q-oreios-nPNG',
       'hinesorum-sPNG', 'irenicus', 'oreios-ID', 'punctulatus',
       'torresiensis'], dtype=object)

In [17]:
# Only the first 29 or so contigs have enough data to make this worthwhile

contigs_list = dict(list(filtered_contigs.items())[:29])

In [20]:
popa = 'farauti_ss-QLD-Mainland'
popb = 'farauti_ss-TSI-sNG'

mapper = GenomePositionMapper(filtered_contigs)

gene_df = mapper.transform_annotations(gene_locs)

list_contig_dfs = []

for contig in contigs_list.keys():
    fst, x = fst_gwss(sample_query_a = f'analysis_cohort == "{popa}"',
             sample_query_b = f'analysis_cohort == "{popb}"',
             window_size=3000,
            contig=contig,
            zarr_base_path = zarr_p,
            df_samples=df_samples)

    list_contig_dfs.append(pd.DataFrame({'contig': contig, 'pos': x, 'fst': fst}))
    
fst_df = pd.concat(list_contig_dfs)
    


In [21]:
fst_df

,contig,pos,fst
0,KI915040,7270.652667,0.282914
1,KI915040,21123.218000,0.144671
2,KI915040,34289.885000,0.371983
3,KI915040,46765.045667,0.278094
4,KI915040,58571.729000,0.334234
...,...,...,...
42,KI915068,750954.251667,0.268009
43,KI915068,769328.138667,0.351695
44,KI915068,788168.650333,0.298223
45,KI915068,803340.299667,0.185077


In [49]:
contig

'KI915069'

In [ ]:
# Let's start

def plot_fst_gwss(
    fst_df
)